In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchvision.models.resnet import ResNet, BasicBlock
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


# Group 1

In [3]:
cat_class = 3
dog_class = 5

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
def filter_cat_vs_dog(dataset, cat_class=3, dog_class=5):
    indices = [i for i, label in enumerate(dataset.targets) if label in [cat_class, dog_class]]
    new_targets = [1 if dataset.targets[i] == cat_class else 0 for i in indices]
    subset = Subset(dataset, indices)
    subset.targets = new_targets
    return subset

In [5]:
class CatDogDataset(Dataset):
    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, _ = self.subset[idx]
        label = self.subset.targets[idx]
        return img, label

In [6]:
train_dataset = CatDogDataset(filter_cat_vs_dog(train_dataset_full))
test_dataset = CatDogDataset(filter_cat_vs_dog(test_dataset_full))

In [7]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

train - test dataset ready.


# Group 2

In [9]:
# --- Part 2: Configure Model ---
# Task 2: Load pretrained ResNet18
print("\nLoading pretrained ResNet18 model...")
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Task 3: Modify the final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
# Task 4: Freeze all layers except layer3, layer4, and fc
# First, freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the layers for fine-tuning
print("Unfreezing layers for fine-tuning: layer3, layer4, and fc.")
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

#Verification(rishab likes goth chicks)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nModel setup is complete.")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print("\nNotebook is now ready for Group 3 (Fine-Tuning).")


Loading pretrained ResNet18 model...
Unfreezing layers for fine-tuning: layer3, layer4, and fc.

Model setup is complete.
  Total parameters:     11,177,538
  Trainable parameters: 10,494,466

Notebook is now ready for Group 3 (Fine-Tuning).


# Group 3